Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [20]:
df.write \
    .mode("overwrite") \
    .parquet("/content/input_parquet")

print("Parquet file created successfully")

Parquet file created successfully


In [21]:
df_parquet = spark.read.parquet("/content/input_parquet")

df_filtered = df_parquet.filter(
    col("user_id").isNotNull()
)

df_filtered.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/content/q12_output")

df_filtered.show(5)

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|product_id| old_name|   category|base_price|quantity|   status|region|priority|user_id|  price| amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13|1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47|8150.82|
|    P00003|Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33|3873.31|
|    P00004|    Novel|Electronics|    594.61|       1|Completed| South|  Medium|  U0850| 594.61| 594.61|
|    P00005|  T-Shirt|   Clothing|   1075.16|       2|Cancelled| North|    High|  U0246|1075.16|2150.32|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
only showing top 5 rows


The Parquet read/write operation was performed in Google Colab because the local Windows Jupyter environment encountered a Hadoop configuration (HADOOP_HOME) issue while handling file write operations. Google Colab provided a Linux-based environment where the Spark pipeline could be executed successfully without the Windows-specific Hadoop configuration issue. This allowed the Parquet file to be processed, null values to be filtered, and the final output to be saved successfully in CSV format.


In [14]:
from google.colab import files

uploaded = files.upload()

Saving source_dataset.csv to source_dataset (1).csv


In [15]:
from pyspark.sql.functions import col

# Read
pipeline_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/source_dataset.csv")

# Transformed
pipeline_df = pipeline_df \
    .withColumnRenamed("old_name", "product_name") \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("final_price", col("base_price") * 1.18)

# Handled nulls
pipeline_df = pipeline_df.filter(
    col("user_id").isNotNull()
)

# Filtered completed transactions
pipeline_df = pipeline_df.filter(
    col("status") == "Completed"
)

# Checked result
pipeline_df.show(10)

+----------+------------+-----------+----------+--------+---------+------+--------+-------+-------+--------+------------------+
|product_id|product_name|   category|base_price|quantity|   status|region|priority|user_id|  price|  amount|       final_price|
+----------+------------+-----------+----------+--------+---------+------+--------+-------+-------+--------+------------------+
|    P00001|       Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13| 1080.26|          637.3534|
|    P00002|       Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47| 8150.82|         1602.9946|
|    P00003|   Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33| 3873.31|          652.9294|
|    P00004|       Novel|Electronics|    594.61|       1|Completed| South|  Medium|  U0850| 594.61|  594.61| 701.6397999999999|
|    P00007|   Face Wash|     Beauty|     110.7|       6|Completed|  East|    High|  U0083|  110.7|   66

In [16]:
# Save as CSV
pipeline_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/content/processed_csv")

# Save as Parquet
pipeline_df.write \
    .mode("overwrite") \
    .parquet("/content/processed_parquet")

print("Pipeline executed successfully.")

Pipeline executed successfully.


### Note on pipeline execution

The initial Spark operations were performed locally using Jupyter Notebook on Windows. However, while executing the final data pipeline, the local Spark environment encountered a Hadoop configuration issue (HADOOP_HOME) during file write operations.

To complete and verify the required **read → transform → filter → write** pipeline, the final execution was performed using Google Colab, where the processed data was successfully saved and verified in both **CSV and Parquet formats**.
